# 02 · Llama 3.2 1B-Instruct → ExecuTorch (.pte) + grounded advisory prompt
**Bharat AI-SoC 2026–27 · PS1** — the challenge requires a Llama-family model to run the advisory on the device through ExecuTorch.

Runs on **Kaggle** (recommended — ~29 GB RAM makes the export easy) or Colab.

**Before you start:** accept the Llama 3.2 licence at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct and create a *Read* token (HF ▸ Settings ▸ Access Tokens).

**Kaggle setup:**
1. *Create ▸ New Notebook ▸ File ▸ Import Notebook* and upload this file.
2. Right panel ▸ *Session options*: **Accelerator → None** (CPU is enough) and **Internet → On**.
3. Top menu ▸ *Add-ons ▸ Secrets* ▸ add `HF_TOKEN` = your token and tick it for this notebook.
4. *Run All*. The export step is quiet for 10–20 min. Then download `ps1_llm_outputs.zip` (about 1 GB) from the Output section, or click *Save Version*.

Produces (in the `out` folder, also zipped):
- `llama3_2_1b_8da4w.pte` — Llama 3.2 1B-Instruct, 4-bit weights / 8-bit dynamic activations (8da4w), KV cache, XNNPACK. About 1 GB, so it runs on phones with 6 GB+ RAM.
- `tokenizer.model` — load it in the app next to the .pte.
- `kb.json` — starter knowledge base (disease → treatment); the LLM only phrases advice **from** this, so it can't invent pesticides or doses.
- `prompt_template.txt` — the exact prompt the Android app must build.
- `llm_benchmark.json` — tokens/sec and latency on the notebook CPU, for the report.

**Colab instead?** Add the token as a Colab secret named `HF_TOKEN` (🔑 icon), then *Run all*.

In [ ]:
import sys
if "torch" in sys.modules:
    raise RuntimeError("PyTorch is already loaded in this session. Click Run ▸ Restart, then run this cell again "
                       "(installing over a loaded PyTorch mixes old and new versions and breaks the export).")
!pip install -q --default-timeout=120 --retries 10 executorch huggingface_hub
import os, json, time, pathlib
IS_KAGGLE = os.path.exists("/kaggle/working")
OUT_DIR = "/kaggle/working/out" if IS_KAGGLE else "/content/out"; os.makedirs(OUT_DIR, exist_ok=True)
FAST_TEST = os.environ.get("FAST_TEST") == "1"

In [ ]:
# ---- Download the original (Meta-format) checkpoint ---------------------
if FAST_TEST:
    CKPT_DIR = os.environ["FAST_CKPT_DIR"]
else:
    from huggingface_hub import snapshot_download, login
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    login(token=token)
    repo = snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"])
    CKPT_DIR = os.path.join(repo, "original")
print(os.listdir(CKPT_DIR))   # expect consolidated.00.pth, params.json, tokenizer.model

## 1 · Export with `export_llm` (8da4w quantisation, XNNPACK, KV cache)
- `max_context_length 1024`: enough for our ~450-token prompt plus a 300-token answer.
- `max_seq_length 768`: the longest prompt processed in one prefill step.
- The `metadata` line tells the on-device runner which tokens start and end a turn in Llama 3.

In [ ]:
PTE = f"{OUT_DIR}/llama3_2_1b_8da4w.pte"
cfg = f"""
base:
  model_class: llama3_2
  checkpoint: {CKPT_DIR}/consolidated.00.pth
  params: {CKPT_DIR}/params.json
  metadata: '{{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}}'
model:
  use_kv_cache: true
  use_sdpa_with_kv_cache: true
  dtype_override: fp32
export:
  max_seq_length: 768
  max_context_length: 1024
  output_name: {PTE}
quantization:
  qmode: 8da4w
  group_size: 32
  embedding_quantize: "4,32"
backend:
  xnnpack:
    enabled: true
    extended_ops: true
"""
if FAST_TEST:   # smoke test uses a tiny random model with no checkpoint
    cfg = cfg.replace(f"  checkpoint: {CKPT_DIR}/consolidated.00.pth\n", "")
open(f"{OUT_DIR}/export_llama.yaml", "w").write(cfg)
t = time.time()
!python -m executorch.extension.llm.export.export_llm --config {OUT_DIR}/export_llama.yaml
print(f"export took {time.time()-t:.0f}s · {os.path.getsize(PTE)/1e9:.2f} GB")
!cp {CKPT_DIR}/tokenizer.model {OUT_DIR}/tokenizer.model

## 2 · Knowledge base (the facts the LLM is allowed to use)
Starter schema with two entries. **Person B** fills it in for all 38 PlantVillage classes (key = the folder name from `labels.json`) using **ICAR / TNAU Agritech Portal / state agriculture department** sources, and puts the source in `ref`.
Fields named `*_hi` / `*_ta` hold the Hindi and Tamil text shown when the farmer picks that language (the model's Indic output isn't reliable enough to generate these).

In [ ]:
kb = {
  "Tomato___Early_blight": {
    "crop": "Tomato", "disease": "Early blight", "pathogen": "Alternaria solani (fungus)",
    "symptoms": "Brown spots with concentric rings on older, lower leaves; yellowing around spots.",
    "cultural": ["Remove and destroy infected lower leaves", "Avoid overhead irrigation; water at the base",
                 "Mulch to stop soil splashing onto leaves", "Rotate away from tomato/potato/brinjal for 2–3 seasons"],
    "organic": ["Spray a Trichoderma- or Bacillus subtilis-based bio-fungicide as per label"],
    "chemical": ["Mancozeb 75% WP or Chlorothalonil 75% WP — follow the label dose and pre-harvest interval"],
    "escalate_if": "Spots spread to upper leaves or fruit within a week",
    "ref": "TODO: add ICAR / TNAU source URL",
    "summary_hi": "TODO", "summary_ta": "TODO"
  },
  "Tomato___healthy": {
    "crop": "Tomato", "disease": "None (healthy)", "symptoms": "No visible disease.",
    "cultural": ["Keep monitoring weekly", "Maintain balanced fertiliser and drainage"],
    "organic": [], "chemical": [], "escalate_if": "New spots or wilting appear", "ref": "",
    "summary_hi": "TODO", "summary_ta": "TODO"
  }
}
json.dump(kb, open(f"{OUT_DIR}/kb.json", "w"), indent=1, ensure_ascii=False)

## 3 · Prompt template (the Android app must build exactly this string)
Uses the Llama 3 chat format. The vision result, severity and KB entry are placed in the prompt; the system prompt forbids anything not in the KB.

In [ ]:
SYSTEM = ("You are KrishiMitra, an offline crop-health assistant for Indian farmers. "
          "Use ONLY the facts in CONTEXT. Never invent pesticide names or doses; if the context has none, say "
          "'consult your local Krishi Vigyan Kendra'. Answer in simple English in at most 6 short bullet points: "
          "what it is, how serious it is, what to do today, organic option, chemical option (say 'follow label dose'), "
          "when to call an expert.")

def build_prompt(label_raw, confidence, severity_label, severity_frac, region="", season="", question=""):
    e = kb.get(label_raw, {})
    ctx = {
        "diagnosis": f"{e.get('crop','?')} — {e.get('disease', label_raw)}",
        "model_confidence": f"{confidence:.0%}",
        "severity": f"{severity_label} ({severity_frac:.0%} of leaf area affected)",
        "symptoms": e.get("symptoms", ""), "cultural_practices": e.get("cultural", []),
        "organic_options": e.get("organic", []), "chemical_options": e.get("chemical", []),
        "escalate_if": e.get("escalate_if", ""), "region": region, "season": season,
    }
    if confidence < 0.6:
        ctx["note"] = "Low confidence: ask the farmer to retake the photo in daylight, one leaf filling the frame."
    user = "CONTEXT:\n" + json.dumps(ctx, ensure_ascii=False, indent=1) + "\n\n" + (question or "Give me advice for my crop.")
    return ("<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n" + SYSTEM + "<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n" + user + "<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n")

prompt = build_prompt("Tomato___Early_blight", 0.94, "Moderate", 0.17, region="Tamil Nadu", season="Rabi")
open(f"{OUT_DIR}/prompt_template.txt", "w").write(prompt)
print(prompt)

## 4 · Run the .pte with ExecuTorch's runner (same engine the Android app uses)

In [ ]:
import executorch.extension.llm.custom_ops.custom_ops   # registers the fused SDPA/KV-cache kernels the .pte uses
try:
    import executorch.kernels.quantized                  # 4-bit embedding kernels
except ImportError:
    pass
from executorch.extension.llm.runner import TextLLMRunner, GenerationConfig
runner = TextLLMRunner(PTE, f"{OUT_DIR}/tokenizer.model")
pieces, t_first = [], []
t0 = time.perf_counter()
def on_token(tok):
    if not t_first: t_first.append(time.perf_counter() - t0)
    pieces.append(tok); print(tok, end="", flush=True)
run_prompt = "<|begin_of_text|>hello" if FAST_TEST else prompt   # smoke test uses a byte-level stand-in tokenizer
runner.generate(run_prompt, GenerationConfig(echo=False, max_new_tokens=40 if FAST_TEST else 300, temperature=0.2), on_token)
total = time.perf_counter() - t0
bench = {"time_to_first_token_s": t_first[0] if t_first else None, "total_s": total, "tokens": len(pieces),
         "decode_tok_per_s": (len(pieces) - 1) / max(total - (t_first[0] if t_first else 0), 1e-6),
         "pte_size_gb": os.path.getsize(PTE) / 1e9, "note": "notebook x86 CPU — re-measure on the phone for the report"}
print("\n\n", json.dumps(bench, indent=1))
json.dump(bench, open(f"{OUT_DIR}/llm_benchmark.json", "w"), indent=1)

In [ ]:
# Package the outputs for the team (the .pte is ~1 GB, so the zip is too)
import shutil
zip_path = shutil.make_archive(os.path.join(os.path.dirname(OUT_DIR), "ps1_llm_outputs"), "zip", OUT_DIR)
print("zipped →", zip_path, f"({os.path.getsize(zip_path)/1e9:.2f} GB)")
if not FAST_TEST and not IS_KAGGLE:   # Colab: also copy to Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p "/content/drive/MyDrive/bharat-ai-soc-ps1/llm" && cp -v {OUT_DIR}/llama3_2_1b_8da4w.pte {OUT_DIR}/tokenizer.model {OUT_DIR}/*.json {OUT_DIR}/*.txt {OUT_DIR}/*.yaml "/content/drive/MyDrive/bharat-ai-soc-ps1/llm/"